In [ ]:
import pickle
import numpy as np
import tensorflow as tf



# Path to your data file
data_path = '/content/drive/MyDrive/Colab Notebooks/RML2016.10a_dict.pkl'

# Load data
with open(data_path, 'rb') as file:
    Xd = pickle.load(file, encoding='latin1')

# Extract mods and snrs
mods, snrs1 = [sorted(list(set([k[j] for k in Xd.keys()]))) for j in [0,1]]

X = []
lbl = []
train_idx = []
val_idx = []
np.random.seed(2016)
a = 0
#snrs1=[-20,-18,-16,-14,-12,-10,-8,-6,-4,-2]
#snrs1=[0,2,4,6,8,10,12,14,16,18]
# Populate X and lbl
for mod in mods:
    for snr in snrs1:
        X.append(Xd[(mod, snr)])
        for i in range(Xd[(mod, snr)].shape[0]):
            lbl.append((mod, snr))
        train_idx += list(np.random.choice(range(a*1000, (a+1)*1000), size=600, replace=False))
        val_idx += list(np.random.choice(list(set(range(a*1000, (a+1)*1000)) - set(train_idx)), size=200, replace=False))
        a += 1

# Stack X
X = np.vstack(X)

# Define test_idx
n_examples = X.shape[0]
test_idx = list(set(range(0, n_examples)) - set(train_idx) - set(val_idx))
np.random.shuffle(train_idx)
np.random.shuffle(val_idx)
np.random.shuffle(test_idx)

# Split into train, val, and test sets
X_train = X[train_idx]
X_val = X[val_idx]
X_test = X[test_idx]

# Define to_onehot function
def to_onehot(yy):
    yy1 = np.zeros([len(yy), len(mods)])
    yy1[np.arange(len(yy)), yy] = 1
    return yy1

# Generate Y_train, Y_val, and Y_test
Y_train = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), train_idx)))
Y_val = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), val_idx)))
Y_test = to_onehot(list(map(lambda x: mods.index(lbl[x][0]), test_idx)))

# Swap axes for training, validation, and test sets
X_train = X_train.swapaxes(2, 1)
X_val = X_val.swapaxes(2, 1)
X_test = X_test.swapaxes(2, 1)




# Print shapes to verify

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_val.shape)
print(Y_test.shape)


In [ ]:
from tensorflow.keras.layers import Input,GRU, Conv2D, Dense,Add,Subtract, Dropout,Multiply, BatchNormalization, Activation,Lambda, Flatten, Reshape, MaxPooling2D, concatenate
from keras.callbacks import EarlyStopping
X1_train=X_train[:,:,0]
X1_test=X_test[:,:,0]
X1_val=X_val[:,:,0]
X2_train=X_train[:,:,1]
X2_test=X_test[:,:,1]
X2_val=X_val[:,:,1]
X_train=np.expand_dims(X_train,axis=3)
X_test=np.expand_dims(X_test,axis=3)
X_val=np.expand_dims(X_val,axis=3)
print(X1_train.shape)

print(X_test.shape)
classes = mods


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Activation,
    Lambda,
    Multiply,
    Add,
    Subtract,
    Reshape,
    concatenate,
    Conv1D,
    MaxPooling1D,
    Flatten,
)

def cal1(x_degrees):
    x_tensor = tf.convert_to_tensor(x_degrees, dtype=tf.float32)

    w = tf.concat([x_tensor * (i + 1) for i in range(128)], axis=-1)

    y = tf.cos(w )

    return tf.expand_dims(y, axis=-1)  # Expand dimensions to (None, 128, 1)

def cal2(x_degrees):
    x_tensor = tf.convert_to_tensor(x_degrees, dtype=tf.float32)


    w = tf.concat([x_tensor * (i + 1) for i in range(128)], axis=-1)  #

    y = tf.sin(w )

    return tf.expand_dims(y, axis=-1)

def CNN(input_shape=(128, 1), num_classes=11):
    # Define inputs
    input1 = Input(shape=input_shape, name='input1')
    input2 = Input(shape=input_shape, name='input2')
    input=concatenate([input1,input2])
    # Perform initial operations as described
    x1 = Flatten()(input)
    x1= Dense(80, name='fc1')(x1)
    x1 = Activation('relu')(x1)
    x1 = Dense(1, name='fc3')(x1)
    x1 = Activation('linear')(x1)

    cos1= Lambda(cal1)(x1)
    sin1 = Lambda(cal2)(x1)
    x11 = Multiply()([input1, cos1])
    x12 = Multiply()([input2, sin1])
    x21 = Multiply()([input2, cos1])
    x22 = Multiply()([input1, sin1])
    y1 = Add()([x11,x12])
    y2 = Subtract()([x21,x22])
    y1   = Reshape(target_shape=(128, 1), name='reshape1')(y1)
    y2 = Reshape(target_shape=(128, 1), name='reshape2')(y2)
    x11 = concatenate([y1,y2,input1,input2])


    # Start convolutional layers
    # The following layers operate on the concatenated result `x11`

    # First convolutional layer
    #x = Conv1D(filters=265, kernel_size=3, activation='relu')(x11)
    #x = MaxPooling1D(pool_size=2)(x)

    # Second convolutional layer
    #x = Conv1D(filters=80, kernel_size=3, activation='relu')(x)
    #x = MaxPooling1D(pool_size=2)(x)

    # Third convolutional layer
    #x = Conv1D(filters=50, kernel_size=2, activation='relu')(x)

   # x = Conv1D(filters=50, kernel_size=8, activation='relu')(x)
    x=Conv1D(64, kernel_size=3 ,activation='relu')(x11)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x=Conv1D(128, kernel_size=3, activation='relu')(x)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x=Conv1D(256, kernel_size=3, activation='relu')(x)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x= Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=[input1, input2], outputs=output)

    return model

model = CNN()
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
model.summary()


In [ ]:
from tensorflow.keras.models import Model
model=CNN()
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
model.summary()
modtype=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']


In [ ]:
X1_train=np.expand_dims(X1_train,axis=2)
X1_test=np.expand_dims(X1_test,axis=2)
X1_val=np.expand_dims(X1_val,axis=2)
X2_train=np.expand_dims(X2_train,axis=2)
X2_test=np.expand_dims(X2_test,axis=2)
X2_val=np.expand_dims(X2_val,axis=2)
print(X1_train.shape)


In [ ]:
from keras import callbacks
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Reshape,GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks
from tensorflow.keras.callbacks import ModelCheckpoint,ReduceLROnPlateau,EarlyStopping
history=model.fit([X1_train, X2_train], Y_train, epochs=100, batch_size=400,
         validation_data=([X1_val, X2_val], Y_val),callbacks = [
                ModelCheckpoint('/content/drive/MyDrive/Colab Notebooks/mmm1.keras', monitor='val_loss', verbose=1, save_best_only=True, mode='auto'),
                ReduceLROnPlateau(monitor='val_loss',factor=0.5,verbose=1,patince=5,min_lr=0.0000001),
                EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='auto'),
                ])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.savefig('CNN2D2.png')  # Save the figure to a file
plt.grid(True)



In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('CNN2D3.png')  # Save the figure to a file
plt.show()

In [ ]:
model.save('/content/drive/MyDrive/Colab Notebooks/CNN_2.keras')

In [ ]:
def calculate_confusion_matrix(Y,Y_hat,classes):
    n_classes = len(classes)
    conf = np.zeros([n_classes,n_classes])
    confnorm = np.zeros([n_classes,n_classes])

    for k in range(0,Y.shape[0]):
        i = list(Y[k,:]).index(1)
        j = int(np.argmax(Y_hat[k,:]))
        conf[i,j] = conf[i,j] + 1

    for i in range(0,n_classes):
        confnorm[i,:] = conf[i,:] / np.sum(conf[i,:])
    # print(confnorm)

    right = np.sum(np.diag(conf))
    wrong = np.sum(conf) - right
    return confnorm,right,wrong

In [ ]:
input1 = np.array(X1_train[0]).reshape(1, 128, 1)  # Shape: (1, 128, 1)
input2 = np.array(X2_train[0]).reshape(1, 128, 1)  # Shape: (1, 128, 1

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

In [ ]:
train_accuracy = history.history['accuracy']
val_accuracy= history.history['val_accuracy']

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_loss) + 1)

# Plotting loss
plt.figure(figsize=(6, 3))
plt.plot(epochs, train_loss, 'r', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Plotting MAE if available
if 'accuracy' in history.history:
    plt.figure(figsize=(6, 3))
    plt.plot(epochs, train_accuracy, 'r', label='Training accuracy')
    plt.plot(epochs, val_accuracy, 'b', label='Validation accuracy')
    plt.title('Training and validation accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv

def plot_confusion_matrix(cm, title='Confusion matrix', cmap=plt.get_cmap("PuRd"), labels=[], save_filename=None):
    plt.figure(figsize=(6, 6))
    plt.imshow(cm * 100, interpolation='nearest', cmap=cmap)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=90, size=12)
    plt.yticks(tick_marks, labels, size=12)

    for i in range(len(tick_marks)):
        for j in range(len(tick_marks)):
            value = int(np.around(cm[i, j] * 100))
            if i != j:
                text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=10)
            elif i == j:
                if value == 100:
                    text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=7, color="white")
                else:
                    text = plt.text(j, i, f"{value}%", ha="center", va="center", fontsize=10, color="white")

    plt.tight_layout()
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv
#snrs1=[0,2,4,6,8,10,12,14,16,18]

classes = mods
modtype=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
model.load_weights('/content/drive/MyDrive/Colab Notebooks/freq_Marah_final_CNNONLY_highsnr.weights.h5')
test_Y_hat = model.predict([X1_test, X2_test])
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
acc2 = {}
acc_mod_snr = np.zeros( (len(classes),len(snrs1)) )
i = 0
for snr in snrs1:
        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_X1_i = X1_test[np.where(np.array(test_SNRs) == snr)]
        test_X2_i = X2_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i_hat = model.predict([test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc2[snr] = 1.0 * cor / (cor + ncor)

        plt.figure(figsize=(8, 6))
        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.tight_layout()
        plt.show()  # Display the plot

        plt.close()
        acc_mod_snr[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1







In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv
#snrs1=[0,2,4,6,8,10,12,14,16,18]

classes = mods
modtype=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
model.load_weights('/content/drive/MyDrive/Colab Notebooks/marahfinalCNNlow.weights.h5')
test_Y_hat = model.predict([X1_test, X2_test])
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
acc3 = {}
acc_mod_snr2 = np.zeros( (len(classes),len(snrs1)) )
i = 0
for snr in snrs1:

        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_X1_i = X1_test[np.where(np.array(test_SNRs) == snr)]
        test_X2_i = X2_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]

        # Estimate classes

        # time_start = time.time()
        test_Y_i_hat = model.predict([test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc3[snr] = 1.0 * cor / (cor + ncor)
        #result = cor / (cor + ncor)

        plt.figure(figsize=(8, 6))
        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.tight_layout()
        plt.show()  # Display the plot

        plt.close()
        acc_mod_snr2[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
acc_mod_snr.shape
result = acc_mod_snr2[:, :10]
result2 =acc_mod_snr[:, 10:]

In [ ]:
result3 = np.concatenate((result, result2), axis=1)
result3.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs1, result3[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah.png')
plt.show()
plt.close()


In [ ]:
m=list(map(lambda x: acc2[x], snrs1))
m1= m[10:]
m2=list(map(lambda x: acc3[x], snrs1))
m3= m2[:10]
a = np.array([m3, m1])
b=a.reshape(20)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))  # Example size, adjust as needed
# Plot accuracy curve
plt.plot(snrs1, b, marker='o', linestyle='-')

# Annotate each point
for snr, accuracy in zip(snrs1, b):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0, 10),
                 ha='center')

plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy on RadioML 2016.10 Alpha")
plt.tight_layout()
plt.grid()
plt.show()  # Display the plot



In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Activation,
    Lambda,
    Multiply,
    Add,
    Subtract,
    Reshape,
    concatenate,
    Conv1D,
    MaxPooling1D,
    Flatten,
)


def CNN(input_shape=(128, 1), num_classes=11):
    # Define inputs
    input1 = Input(shape=input_shape, name='input1')
    input2 = Input(shape=input_shape, name='input2')
    input=concatenate([input1,input2])



    # Start convolutional layers
    # The following layers operate on the concatenated result `x11`

    # First convolutional layer
    #x = Conv1D(filters=265, kernel_size=3, activation='relu')(x11)
    #x = MaxPooling1D(pool_size=2)(x)

    # Second convolutional layer
    #x = Conv1D(filters=80, kernel_size=3, activation='relu')(x)
    #x = MaxPooling1D(pool_size=2)(x)

    # Third convolutional layer
    #x = Conv1D(filters=50, kernel_size=2, activation='relu')(x)

   # x = Conv1D(filters=50, kernel_size=8, activation='relu')(x)
    x=Conv1D(64, kernel_size=3 ,activation='relu')(input)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x=Conv1D(128, kernel_size=3, activation='relu')(x)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x=Conv1D(256, kernel_size=3, activation='relu')(x)
    x=MaxPooling1D(pool_size=2)(x)
    x=Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)
    x= Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)

    # Create model
    model = Model(inputs=[input1, input2], outputs=output)

    return model

# Example usage
model2= CNN()
model2.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')
model2.summary()


In [ ]:
model2.load_weights('/content/drive/MyDrive/Colab Notebooks/marahfinalCNN.weights.h5')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import csv
#snrs1=[0,2,4,6,8,10,12,14,16,18]

classes=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM']
test_Y_hat = model2.predict([X1_test, X2_test])
confnorm, _, _ = calculate_confusion_matrix(Y_test, test_Y_hat, classes)
acc3 = {}
acc_mod_snr2 = np.zeros( (len(classes),len(snrs1)) )
i = 0
for snr in snrs1:

        test_SNRs = [lbl[x][1] for x in test_idx]
        test_X_i=X_test[np.where(np.array(test_SNRs) == snr)]
        test_X1_i = X1_test[np.where(np.array(test_SNRs) == snr)]
        test_X2_i = X2_test[np.where(np.array(test_SNRs) == snr)]
        test_Y_i = Y_test[np.where(np.array(test_SNRs) == snr)]

        # Estimate classes

        # time_start = time.time()
        test_Y_i_hat = model2.predict([test_X1_i,test_X2_i])

        confnorm_i,cor,ncor = calculate_confusion_matrix(test_Y_i,test_Y_i_hat,classes)
        acc3[snr] = 1.0 * cor / (cor + ncor)

        plt.figure(figsize=(8, 6))
        plot_confusion_matrix(confnorm_i, labels=['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','4-PAM','16-QAM','64-QAM','QPSK','WBFM'])
        plt.tight_layout()
        plt.show()

        plt.close()
        acc_mod_snr2[:,i] = np.round(np.diag(confnorm_i)/np.sum(confnorm_i,axis=1),3)
        i = i +1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_mod = 11

plt.figure(figsize=(8, 6))
plt.xlabel("Signal to Noise Ratio")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy for Each Mod")

for i in range(0, num_mod):
    plt.plot(snrs1, acc_mod_snr2[i], label=classes[i])

# Add legend, grid, save, and show plot
plt.legend()
plt.grid()
plt.savefig('marah.png')
plt.show()
plt.close()


In [ ]:
# Plot Accuracy vs. SNR
plt.figure(figsize=(10, 6))
plt.plot(snrs1, b, marker='o', linestyle='-', color='orange', label='after')

# Annotate each point with its value
for snr, accuracy in zip(snrs1, b):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')
plt.plot(snrs1, list(map(lambda x: acc3[x], snrs1)), marker='o', linestyle='-', color='b', label='before')

for snr, accuracy in zip(snrs1, list(map(lambda x: acc3[x], snrs1))):
    plt.annotate(f'{accuracy:.2f}',
                 (snr, accuracy),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center')
plt.xlabel("Signal to Noise Ratio (SNR)")
plt.ylabel("Classification Accuracy")
plt.title("Classification Accuracy vs. SNR")
plt.grid(True)
plt.legend()
plt.savefig('figure1.png')  # Save the figure to a file
plt.show()

In [ ]:
z=list(map(lambda x: acc3[x], snrs1))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sample data
snrs = snrs1  # Signal to Noise Ratios
frequency = b-z


# Number of groups
n_groups = len(snrs)

# Create a plot
fig, ax = plt.subplots()
index = np.arange(n_groups)
bar_width = 0.25
#plt.figure(figsize=(6, 4))

# Bar plots
rects1 = ax.bar(index, frequency, bar_width, label='accuracy_gain',color='blue')

# Labels, Title and Custom x-axis
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Accuracy Gain ')
ax.set_title('Accuracy Gain by SNR')
ax.set_xticks(index + bar_width)
ax.set_xticklabels(snrs)
ax.legend()
plt.grid()

# Display plot
plt.show()

